In [ ]:
!pip install transformers torch pandas openpyxl -q

In [ ]:
import pandas as pd
import numpy as np
from transformers import pipeline

In [ ]:
df = pd.read_excel("../Data/Social_Media_Posting_Report_ANONYMIZED.xlsx")
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Students: {df['student_id'].nunique()}")
print(f"Posts per student:\n{df['student_id'].value_counts().sort_index()}")

In [ ]:
q5_col = [c for c in df.columns if c.startswith("Q5")][0]
df = df.rename(columns={q5_col: "Q5_caption"})

# Clean the caption text
def clean_text(x):
    if pd.isna(x): return ""
    s = str(x).strip()
    return "" if s.upper() == "NA" else s

df["content_text"] = df["Q5_caption"].apply(clean_text)
df["has_text"] = df["content_text"].str.len() > 0
print(f"Usable rows: {df['has_text'].sum()} / {len(df)}")
print(f"Caption length stats: min={df.loc[df['has_text'],'content_text'].str.len().min()}, max={df.loc[df['has_text'],'content_text'].str.len().max()}, median={int(df.loc[df['has_text'],'content_text'].str.len().median())}")

In [ ]:
emotion_model = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,
    truncation=True
)

# Sanity check on one sentence
print(emotion_model("I'm so excited for the Super Bowl!"))

In [ ]:
texts = df.loc[df["has_text"], "content_text"].tolist()
texts = [t[:1500] for t in texts]   # safety trim; model truncates at 512 tokens anyway

print(f"Running emotion model on {len(texts)} posts...")
results = emotion_model(texts, batch_size=16)
print(f"Got {len(results)} results")

In [ ]:
emotion_labels = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
emo_cols = [f"emo_{c}" for c in emotion_labels]

# Drop existing emotion columns if they exist (safe re-run)
df = df.drop(columns=[c for c in emo_cols + ["dominant_emotion", "dominant_emotion_score"] if c in df.columns])

# Build emotion dataframe from results
rows = [{item["label"]: item["score"] for item in res} for res in results]
emo_df = pd.DataFrame(rows)[emotion_labels]
emo_df.columns = emo_cols
emo_df.index = df.index[df["has_text"]]

# Join the scores back to df
df = df.join(emo_df)

# Compute dominant emotion only on rows with text
df["dominant_emotion"] = None
df["dominant_emotion_score"] = None
df.loc[df["has_text"], "dominant_emotion"] = (
    df.loc[df["has_text"], emo_cols].idxmax(axis=1).str.replace("emo_", "")
)
df.loc[df["has_text"], "dominant_emotion_score"] = df.loc[df["has_text"], emo_cols].max(axis=1)

print("Done. Sample:")
print(df[["student_id", "dominant_emotion", "dominant_emotion_score"]].head())

In [ ]:
print("=== Dominant emotion counts ===")
print(df["dominant_emotion"].value_counts(dropna=False))
print()
print("=== Mean scores across all posts ===")
print(df[emo_cols].mean().sort_values(ascending=False))
print()
print("=== Per-student dominant emotion breakdown ===")
print(pd.crosstab(df["student_id"], df["dominant_emotion"]))

In [ ]:
for emo in emotion_labels:
    subset = df[df["dominant_emotion"] == emo]
    print(f"\n=== {emo.upper()} ({len(subset)} posts) ===")
    if len(subset) == 0: continue
    top = subset.nlargest(3, f"emo_{emo}")
    for _, r in top.iterrows():
        print(f"  [{r[f'emo_{emo}']:.2f}] {r['content_text'][:180]}")

In [ ]:
df.to_csv("emotion_output_709_anonymized.csv", index=False)
print("Saved emotion_output_709_anonymized.csv")

# Emotion Inference — Conclusion

## What we found

The model ran cleanly on 703 of 709 posts. The overall distribution makes sense for a social media feed: 36% neutral, 25% joy, 20% fear, with anger, sadness, surprise, and disgust making up the rest.

The most interesting finding is at the student level. Students have **measurably different emotional diets**. S07's feed is 70% neutral content. S01's feed is 51% fear and anger. S08 and S09 lean heavily toward joy. This variance is exactly the kind of signal the project is supposed to surface, because different students are being exposed to genuinely different kinds of content.

## Where the model struggles

Three failure patterns showed up in the sanity check:

1. **Keyword reactivity.** Posts get flagged based on emotion words in them, even when the tone is playful, ironic, or news-reporting. A relatable fridge organization video tagged as 99% anger because of the hashtag `#ragebait`.

2. **Meta-content confusion.** Articles or self-help posts *about* an emotion get flagged as expressing that emotion. A motivational post titled "The fear of enjoying your life" was scored as 99% fear.

3. **Short captions.** Posts with very little text (a question mark, just hashtags) default to high-confidence guesses on whatever the model can grab.

## Bottom line

The pipeline works. The per-post labels are usable but noisy and should be interpreted with these limitations in mind. The student-level patterns are likely real because individual misclassifications wash out when aggregated across ~60 posts per student.

Three options for handling the noise going forward:
- Apply a confidence threshold (only trust high-confidence calls)
- Aggregate and accept noise (rely on averages, not individual posts)
- Try a different model, for example the lab's custom checkpoint if available

Decision to be made with Yiqi in the next meeting.